In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GroupedQueryAttention(nn.Module):
    def __init__(self, embed_dim: int, num_q_heads: int, num_kv_heads: int, head_dim: int):
        """
        初始化 GQA 模块。
        Args:
            embed_dim (int): 输入 embedding 的维度。
            num_q_heads (int): 查询头的数量。
            num_kv_heads (int): 键/值头的数量。num_q_heads 必须是 num_kv_heads 的整数倍。
            head_dim (int): 每个头的维度。
        """
        super().__init__()
        if num_q_heads % num_kv_heads != 0:
            raise ValueError("num_q_heads 必须是 num_kv_heads 的整数倍")

        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        self.num_groups = num_q_heads // num_kv_heads # 每个 K/V 头对应的 Q 头数量

        # Q, K, V 的线性投影层
        # Q 投影到 (num_q_heads * head_dim)
        self.W_q = nn.Linear(embed_dim, num_q_heads * head_dim, bias = False)
        # K 投影到 (num_kv_heads * head_dim)
        self.W_k = nn.Linear(embed_dim, num_kv_heads * head_dim, bias = False)
        # V 投影到 (num_kv_heads * head_dim)
        self.W_v = nn.Linear(embed_dim, num_kv_heads * head_dim, bias = False)

        # 输出投影层
        self.W_o = nn.Linear(num_q_heads * head_dim, embed_dim, bias = False)

        # RoPE (通常在 Attention 模块外部或内部对 Q 和 K 应用)
        # 这里为了演示，我们假设 RoPE 会在 GQA 外部应用，或者在forward中传入处理好的Q,K
        # self.rope = RoPE(head_dim, max_seq_len=2048) # 示例

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """
        将 K 或 V 张量从 num_kv_heads 扩展到 num_q_heads。
        Args:
            x (torch.Tensor): K 或 V 张量，形状 (batch_size, seq_len, num_kv_heads, head_dim)
        Returns:
            torch.Tensor: 扩展后的张量，形状 (batch_size, seq_len, num_q_heads, head_dim)
        """
        batch_size, seq_len, _, _ = x.shape
        if self.num_groups == 1:
            return x
        # x: (B, S, H_kv, D_h)
        # -> (B, S, H_kv, 1, D_h) 插入一个维度用于重复
        # -> (B, S, H_kv, num_groups, D_h) 重复
        # -> (B, S, H_kv * num_groups, D_h) = (B, S, H_q, D_h) reshape
        x = x.unsqueeze(3).repeat(1, 1, 1, self.num_groups, 1)
        x = x.reshape(batch_size, seq_len, self.num_q_heads, self.head_dim)
        return x

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
                mask: torch.Tensor = None, apply_rope_func=None, current_pos_q=0, current_pos_k=0):
        """
        GQA 前向传播。
        Args:
            query (torch.Tensor): 查询张量，形状 (batch_size, q_seq_len, embed_dim)
            key (torch.Tensor): 键张量，形状 (batch_size, kv_seq_len, embed_dim)
            value (torch.Tensor): 值张量，形状 (batch_size, kv_seq_len, embed_dim)
            mask (torch.Tensor, optional):注意力掩码。形状 (batch_size, 1, q_seq_len, kv_seq_len)
                                           或者能广播到这个形状。
            apply_rope_func (callable, optional): 应用 RoPE 的函数。
                                                  例如 `lambda t, pos: rope_instance(t, pos)`
            current_pos_q (int): query token 的起始位置 (用于RoPE)
            current_pos_k (int): key/value token 的起始位置 (用于RoPE)

        Returns:
            torch.Tensor: GQA 输出，形状 (batch_size, q_seq_len, embed_dim)
        """
        batch_size, q_seq_len, _ = query.shape
        _, kv_seq_len, _ = key.shape

        # 1. 线性投影
        q_proj = self.W_q(query) # (B, q_S, H_q*D_h)
        k_proj = self.W_k(key)   # (B, kv_S, H_kv*D_h)
        v_proj = self.W_v(value) # (B, kv_S, H_kv*D_h)

        # 2. Reshape Q, K, V 为多头形式
        # (B, S, H*D_h) -> (B, S, H, D_h)
        q_heads = q_proj.view(batch_size, q_seq_len, self.num_q_heads, self.head_dim)
        k_heads = k_proj.view(batch_size, kv_seq_len, self.num_kv_heads, self.head_dim)
        v_heads = v_proj.view(batch_size, kv_seq_len, self.num_kv_heads, self.head_dim)

        # 3. 应用 RoPE
        if apply_rope_func:
            q_heads = apply_rope_func(q_heads, current_pos_q)
            k_heads = apply_rope_func(k_heads, current_pos_k)
            # v_heads 不需要 RoPE

        # 4. 如果是 GQA/MQA，扩展 K 和 V 头以匹配 Q 头
        # (B, S, H_kv, D_h) -> (B, S, H_q, D_h)
        k_heads_grouped = self._repeat_kv(k_heads)
        v_heads_grouped = self._repeat_kv(v_heads)

        # 5. 为了计算方便，调整维度顺序 (B, H, S, D_h)
        q_heads = q_heads.transpose(1, 2)           # (B, H_q, q_S, D_h)
        k_heads_grouped = k_heads_grouped.transpose(1, 2) # (B, H_q, kv_S, D_h)
        v_heads_grouped = v_heads_grouped.transpose(1, 2) # (B, H_q, kv_S, D_h)

        # 6. 计算注意力分数 (Scaled Dot-Product Attention)
        # (B, H_q, q_S, D_h) @ (B, H_q, D_h, kv_S) -> (B, H_q, q_S, kv_S)
        scores = torch.matmul(q_heads, k_heads_grouped.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # 7. 应用掩码 (如果提供)
        if mask is not None:
            # mask 通常是 (B, 1, q_S, kv_S) 或 (q_S, kv_S)
            # 需要确保 mask 的维度与 scores 兼容
            # 如果 mask 是 (B, q_S, kv_S), 扩展为 (B, 1, q_S, kv_S)
            if mask.ndim == 3 and mask.shape[0] == batch_size and mask.shape[1] == q_seq_len and mask.shape[2] == kv_seq_len:
                 mask = mask.unsqueeze(1) # (B, 1, q_S, kv_S)
            scores = scores.masked_fill(mask == 0, float('-inf')) # mask中0的位置是需要被屏蔽的

        # 8. Softmax
        attention_weights = F.softmax(scores, dim=-1) # (B, H_q, q_S, kv_S)

        # 9. 加权求和 V
        # (B, H_q, q_S, kv_S) @ (B, H_q, kv_S, D_h) -> (B, H_q, q_S, D_h)
        output = torch.matmul(attention_weights, v_heads_grouped)

        # 10. 合并头 (Concat)
        # (B, H_q, q_S, D_h) -> (B, q_S, H_q, D_h) -> (B, q_S, H_q*D_h)
        output = output.transpose(1, 2).contiguous().view(batch_size, q_seq_len, -1)

        # 11. 输出投影
        output = self.W_o(output) # (B, q_S, embed_dim)

        return output


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class RoPE(nn.Module):
    def __init__(self, dim:int, max_seq_len: int, theta: float = 10000.0):
        """
        初始化 RoPE 模块。
        Args:
            dim (int): 输入 embeddings 的最后一个维度的大小 (通常是 head_dim)。
                       注意：dim 必须是偶数，因为 RoPE 成对操作维度。
            max_seq_len (int): 模型能处理的最大序列长度。
            theta (float): RoPE 中的 base value，用于计算频率。
        """
        super().__init__()
        if dim % 2 != 0:
            raise ValueError("dim 必须是偶数，因为 RoPE 成对操作维度。")

        self.dim = dim
        self.max_seq_len = max_seq_len
        self.theta = theta

        # freqs shape: (dim / 2)
        # freqs_i = 1 / (theta ^ (2i/dim) )
        # freqs: [freqs_0, freqs_1, ..., freqs_{dim/2 - 1}]
        freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))

        # 位置t
        # t shape: (max_seq_len)
        # t: [0, 1, ..., max_seq_len - 1]
        t = torch.arange(max_seq_len, device = freqs.device)

        # cal m * freqs_i
        # freqs_outer shape: (max_seq_len, dim / 2)
        # freqs_outer[m, i] = m * freqs_i
        freqs_outer = torch.outer(t, freqs)

        # trans m * freqs_i to 复数形式: cos(m * theta_i) + j * sin(m * theta_i)
        # freqs_cis shape: (max_seq_len, dim / 2)
        # 每个元素是一个复数 e^(j * m * freqs_i)
        freqs_cis = torch.polar(torch.ones_like(freqs_outer), freqs_outer)

        # 注册为 buffer，这样它会被移动到正确的设备并且不会被视为模型参数
        self.register_buffer("freqs_cis", freqs_cis)

    def forward(self, x: torch.Tensor, current_pos: int = 0):
        """
        对输入张量 x 应用 RoPE。
        Args:
            x (torch.Tensor): 输入张量，形状通常是 (batch_size, num_heads, seq_len, head_dim)
                              或者 (batch_size, seq_len, num_heads, head_dim)
                              或者任何最后一个维度是 self.dim 的张量。
                              这里我们假设最后一个维度是 head_dim。
            current_pos (int): 当前生成token的位置（用于KV Cache的场景）。
                               如果处理整个序列，一般为0。
        Returns:
            torch.Tensor: 应用了 RoPE 的张量，形状与 x 相同。
        """

        # 取当前序列对应的旋转因子
        seq_len = x.shape[-2] # 假设倒数第二个维度是序列长度
        # self.freqs_cis形状：(max_seq_len, dim//2)
        freqs_cis_to_apply = self.freqs_cis[current_pos : current_pos + seq_len, :]

        # 把 x 变成复数
        # x shape: (B, H, S, D)
        # x_complex shape: (B, H, S, D/2)
        x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))


        # 对齐形状，准备广播
        # shape: (S, D/2) -> (1, 1, S, D/2)
        freqs_cis_reshaped = freqs_cis_to_apply.unsqueeze(0).unsqueeze(0)

        # 复数乘法 = 旋转
        x_rotated = x_complex * freqs_cis_reshaped

        # 变回实数
        # 复数 → 实部和虚部拆开：[..., real, imag]
        # 形状： (B, H, S, D/2, 2)
        x_out = torch.view_as_real(x_rotated)

        # reshape 回原形状
        # 从 (..., D/2, 2)→ (..., D)
        x_out = x_out.reshape(*x.shape)

        return x_out.type_as(x)



In [11]:
def example_gqa():
    print("--- GQA Example ---")
    batch_size = 2
    q_seq_len = 5  # 查询序列长度
    kv_seq_len = 5 # 键/值序列长度 (自注意力时与q_seq_len相同，交叉注意力时可能不同)
    embed_dim = 128 # 输入 embedding 维度

    # GQA 配置
    num_q_heads = 8  # Query heads
    num_kv_heads = 2 # Key/Value heads (num_q_heads % num_kv_heads == 0)
    head_dim = embed_dim // num_q_heads # 通常 embed_dim = num_q_heads * head_dim
                                       # 但这里为了GQA的灵活性，不一定严格相等
                                       # 如果要严格，embed_dim应该能被num_q_heads整除
                                       # 实际 Llama 中 embed_dim 就是 num_q_heads * head_dim

    print(f"GQA Config: embed_dim={embed_dim}, num_q_heads={num_q_heads}, num_kv_heads={num_kv_heads}, head_dim={head_dim}")
    if embed_dim % num_q_heads != 0 :
        print(f"Warning: embed_dim ({embed_dim}) is not perfectly divisible by num_q_heads ({num_q_heads}). Head_dim is {head_dim}.")
        # 重新计算 head_dim，使 embed_dim = num_q_heads * head_dim
        # head_dim = embed_dim // num_q_heads
        # print(f"Adjusted head_dim to: {head_dim}")
        # 如果坚持使用原始的 head_dim，那么 W_o 的输出维度可能和 embed_dim 不匹配
        # 或者 W_q 的输入维度和 embed_dim 不匹配
        # 这里我们的实现中，W_q 输入是 embed_dim, 输出是 num_q_heads * head_dim
        # W_o 输入是 num_q_heads * head_dim, 输出是 embed_dim

    # 创建 GQA 实例
    gqa_layer = GroupedQueryAttention(embed_dim, num_q_heads, num_kv_heads, head_dim)

    # 模拟 RoPE (在 GQA 外部或作为参数传入)
    rope_instance = RoPE(dim=head_dim, max_seq_len=max(q_seq_len, kv_seq_len) + 10)
    def apply_rope_to_q_k(tensor, pos):
        # RoPE 期望 (..., S, D), GQA的head张量是 (B, S, H, D_h)
        # 需要调整一下，或者 RoPE 能够处理 (B, S, H, D_h)
        # 我们的 RoPE 实现可以处理最后一个维度是 head_dim 的情况
        # (B, H, S, D_h) -> (B,S,H,D_h) -> RoPE -> (B,S,H,D_h) -> (B,H,S,D_h)
        # 或者直接对 (B, H, S, D_h) 操作，只要 RoPE 的 forward 能正确索引 freqs_cis
        # 当前 RoPE 实现假设倒数第二个维度是 seq_len，最后一个是 head_dim
        # q_heads/k_heads (B, S, H, D_h)
        return rope_instance(tensor, current_pos=pos)


    # 创建模拟输入 (通常来自 embedding 层和上一个 transformer block)
    query = torch.randn(batch_size, q_seq_len, embed_dim)
    key   = torch.randn(batch_size, kv_seq_len, embed_dim) # 在真实场景中，K, V 可能来自encoder或之前的解码步
    value = torch.randn(batch_size, kv_seq_len, embed_dim)

    # 创建一个简单的上三角掩码 (用于自回归解码)
    # mask 形状 (q_seq_len, kv_seq_len)
    # 如果 q_seq_len != kv_seq_len (例如 KV cache)，mask 需要更小心
    # 这里假设自注意力，q_seq_len == kv_seq_len
    if q_seq_len == kv_seq_len:
        attn_mask = torch.triu(torch.ones(q_seq_len, kv_seq_len), diagonal=1).bool()
        # unsqueeze for batch and head dim: (1, 1, q_S, kv_S)
        # GQA forward 会处理 (B, q_S, kv_S) -> (B, 1, q_S, kv_S)
        attn_mask = attn_mask.unsqueeze(0).repeat(batch_size, 1, 1) # (B, q_S, kv_S)
    else: # 例如 kv_cache, q_seq_len=1, kv_seq_len=N
        attn_mask = None # 无掩码或特定掩码

    print(f"Input query shape: {query.shape}")
    print(f"Input key shape: {key.shape}")
    print(f"Input value shape: {value.shape}")
    if attn_mask is not None:
        print(f"Attention mask shape: {attn_mask.shape}")


    # 前向传播
    # output = gqa_layer(query, key, value, mask=attn_mask, apply_rope_func=apply_rope_to_q_k)
    # 为了简化，这里 apply_rope_func 我们在GQA内部直接调用。
    # 修改GQA的forward，使得它直接能处理 (B,S,H,D_h) -> (B,H,S,D_h)
    # 或者修改 apply_rope_func
    # GQA的forward中 q_heads (B,S,H,D_h) -> transpose -> (B,H,S,D_h)
    # RoPE(q_heads_transposed)

    # 外部应用 RoPE 的方式：
    q_proj_pre_rope = gqa_layer.W_q(query).view(batch_size, q_seq_len, gqa_layer.num_q_heads, gqa_layer.head_dim)
    k_proj_pre_rope = gqa_layer.W_k(key).view(batch_size, kv_seq_len, gqa_layer.num_kv_heads, gqa_layer.head_dim)

    # RoPE 作用于 (B, S, H, D_h)
    # 注意 RoPE 的 current_pos 参数，这里假设从0开始
    q_proj_post_rope = rope_instance(q_proj_pre_rope, current_pos=0)
    k_proj_post_rope = rope_instance(k_proj_pre_rope, current_pos=0)

    # 准备v_proj
    v_proj = gqa_layer.W_v(value).view(batch_size, kv_seq_len, gqa_layer.num_kv_heads, gqa_layer.head_dim)

    # 调整维度顺序以适应 scaled_dot_product_attention
    # (B, S, H, D) -> (B, H, S, D)
    q_final = q_proj_post_rope.transpose(1, 2)
    k_final = k_proj_post_rope.transpose(1, 2)
    v_final = v_proj.transpose(1, 2)

    # 手动实现 GQA 的 K,V 重复部分 for F.scaled_dot_product_attention
    if gqa_layer.num_groups > 1:
        # k_final: (B, H_kv, S, D_h)
        # v_final: (B, H_kv, S, D_h)
        k_final = k_final.unsqueeze(2).repeat(1, 1, gqa_layer.num_groups, 1, 1).flatten(1,2)
        v_final = v_final.unsqueeze(2).repeat(1, 1, gqa_layer.num_groups, 1, 1).flatten(1,2)

    # 使用 PyTorch 内置的 scaled_dot_product_attention (推荐，更优化)
    # 注意：F.scaled_dot_product_attention 的 mask 是 True 代表屏蔽
    sdpa_mask = None
    if attn_mask is not None:
         sdpa_mask = attn_mask.squeeze(1) # (B, q_S, kv_S)

    # F.scaled_dot_product_attention 期望 (B, H, S, D)
    # q_final: (B, num_q_heads, q_seq_len, head_dim)
    # k_final: (B, num_q_heads, kv_seq_len, head_dim) after repeat
    # v_final: (B, num_q_heads, kv_seq_len, head_dim) after repeat

    # 模拟 GQA 内部的scaled_dot_product_attention
    # (B, H_q, q_S, D_h) @ (B, H_q, D_h, kv_S) -> (B, H_q, q_S, kv_S)
    scores = torch.matmul(q_final, k_final.transpose(-2, -1)) / math.sqrt(gqa_layer.head_dim)
    if attn_mask is not None: # mask 是 0 代表保留，True 代表屏蔽
        # attn_mask 应该是 (B, 1, q_S, kv_S)
        scores = scores.masked_fill(attn_mask.bool().unsqueeze(1), float('-inf'))
    attention_weights = F.softmax(scores, dim=-1)
    # (B, H_q, q_S, kv_S) @ (B, H_q, kv_S, D_h) -> (B, H_q, q_S, D_h)
    context_vector = torch.matmul(attention_weights, v_final)

    # 合并头并输出投影
    context_vector = context_vector.transpose(1,2).contiguous().view(batch_size, q_seq_len, -1)
    output = gqa_layer.W_o(context_vector)

    print(f"GQA output shape: {output.shape}")
    assert output.shape == (batch_size, q_seq_len, embed_dim)

    # 测试 MHA (num_q_heads == num_kv_heads)
    print("\n--- Testing GQA as MHA ---")
    mha_heads = 4
    gqa_as_mha = GroupedQueryAttention(embed_dim, mha_heads, mha_heads, head_dim=embed_dim//mha_heads)
    output_mha = gqa_as_mha(query, key, value, mask=attn_mask, apply_rope_func=None) # RoPE 略
    print(f"MHA-like GQA output shape: {output_mha.shape}")
    assert output_mha.shape == (batch_size, q_seq_len, embed_dim)

    # 测试 MQA (num_kv_heads == 1)
    print("\n--- Testing GQA as MQA ---")
    mqa_q_heads = 8
    gqa_as_mqa = GroupedQueryAttention(embed_dim, mqa_q_heads, 1, head_dim=embed_dim//mqa_q_heads)
    output_mqa = gqa_as_mqa(query, key, value, mask=attn_mask, apply_rope_func=None) # RoPE 略
    print(f"MQA-like GQA output shape: {output_mqa.shape}")
    assert output_mqa.shape == (batch_size, q_seq_len, embed_dim)

    print("GQA example finished.\n")



In [12]:
example_gqa()

--- GQA Example ---
GQA Config: embed_dim=128, num_q_heads=8, num_kv_heads=2, head_dim=16
Input query shape: torch.Size([2, 5, 128])
Input key shape: torch.Size([2, 5, 128])
Input value shape: torch.Size([2, 5, 128])
Attention mask shape: torch.Size([2, 5, 5])
GQA output shape: torch.Size([2, 5, 128])

--- Testing GQA as MHA ---
MHA-like GQA output shape: torch.Size([2, 5, 128])

--- Testing GQA as MQA ---
MQA-like GQA output shape: torch.Size([2, 5, 128])
GQA example finished.

